# 30 — Model Evaluation

## Objetivo

Cerrar la fase experimental comparando de forma científica:

```text
Baseline supervisado
vs
Aprendizaje semisupervisado
```

sobre **exactamente las mismas predicciones OOF y los mismos folds**.

Este notebook no vuelve a entrenar modelos. Su función es interpretar la evidencia obtenida en los notebooks 28 y 29.

El estado heredado es:

```text
H1 → bloqueado
H4 → bloqueado
H5 → bloqueado
H6 → evaluable experimentalmente
```

Para H6 se comparan:

- Random Forest supervisado;
- configuración SSL primaria `PRIMARY_OVERLAP_095_CAP1`;
- configuraciones SSL de sensibilidad.

La pregunta científica central es:

> ¿La incorporación de U mediante pseudo-etiquetado mejora el desempeño respecto al baseline supervisado?

La decisión final distingue:

- mejora;
- equivalencia práctica;
- deterioro;
- resultado no concluyente.

No se usan pseudo-etiquetas como ground truth.


## 1. Configuración

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix
)

try:
    from scipy.stats import binomtest
except ImportError:
    binomtest = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

SEMISUP_INTERIM_DIR = (
    PROJECT_DIR / "data" / "semisupervised" / "interim"
)

SEMISUP_REPORTS_DIR = (
    PROJECT_DIR / "reports" / "tables" / "semisupervised"
)

SEMISUP_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ALL_TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

PRIMARY_CONFIG = "PRIMARY_OVERLAP_095_CAP1"
RANDOM_STATE = 42

print("PROJECT_DIR:", PROJECT_DIR)


## 2. Cargar resultados de notebooks 28 y 29

In [ ]:
paths = {
    "baseline_metrics":
        SEMISUP_REPORTS_DIR
        / "baseline_aggregate_metrics.csv",

    "baseline_oof":
        SEMISUP_REPORTS_DIR
        / "baseline_oof_predictions.csv",

    "baseline_provenance":
        SEMISUP_REPORTS_DIR
        / "baseline_provenance_by_class.csv",

    "baseline_continuation":
        SEMISUP_REPORTS_DIR
        / "baseline_continuation_decision.csv",

    "ssl_metrics":
        SEMISUP_REPORTS_DIR
        / "ssl_aggregate_metrics.csv",

    "ssl_oof":
        SEMISUP_REPORTS_DIR
        / "ssl_oof_predictions.csv",

    "ssl_vs_baseline":
        SEMISUP_REPORTS_DIR
        / "ssl_vs_supervised_baseline.csv",

    "pseudo_summary":
        SEMISUP_REPORTS_DIR
        / "ssl_pseudo_label_summary.csv",

    "pseudo_labels":
        SEMISUP_REPORTS_DIR
        / "ssl_pseudo_labels.csv",

    "pseudo_stability":
        SEMISUP_REPORTS_DIR
        / "ssl_pseudo_label_stability.csv",

    "ssl_training_decision":
        SEMISUP_REPORTS_DIR
        / "ssl_training_decision.csv",

    "target_readiness":
        SEMISUP_REPORTS_DIR
        / "semisupervised_master_v2_target_readiness.csv"
}

missing = [
    str(path)
    for path in paths.values()
    if not path.exists()
]

assert missing == [], missing

baseline_metrics = pd.read_csv(
    paths["baseline_metrics"]
)

baseline_oof = pd.read_csv(
    paths["baseline_oof"]
)

baseline_provenance = pd.read_csv(
    paths["baseline_provenance"]
)

baseline_continuation = pd.read_csv(
    paths["baseline_continuation"]
)

ssl_metrics = pd.read_csv(
    paths["ssl_metrics"]
)

ssl_oof = pd.read_csv(
    paths["ssl_oof"]
)

ssl_vs_baseline = pd.read_csv(
    paths["ssl_vs_baseline"]
)

pseudo_summary = pd.read_csv(
    paths["pseudo_summary"]
)

pseudo_labels = pd.read_csv(
    paths["pseudo_labels"]
)

pseudo_stability = pd.read_csv(
    paths["pseudo_stability"]
)

ssl_training_decision = pd.read_csv(
    paths["ssl_training_decision"]
)

target_readiness = pd.read_csv(
    paths["target_readiness"]
)

print("Baseline metrics:", baseline_metrics.shape)
print("SSL metrics:", ssl_metrics.shape)
print("Baseline OOF:", baseline_oof.shape)
print("SSL OOF:", ssl_oof.shape)

display(ssl_training_decision)


## 3. Funciones de métricas

In [ ]:
def safe_roc_auc(
    y_true,
    y_prob
):

    if np.unique(
        y_true
    ).size < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        y_prob
    )


def compute_metrics(
    y_true,
    y_pred,
    y_prob
):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "n":
            len(y_true),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "precision_1":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall_1":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1_1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "roc_auc":
            safe_roc_auc(
                y_true,
                y_prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                y_prob
            ),

        "brier":
            brier_score_loss(
                y_true,
                y_prob
            ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


## 4. Estado de targets

No se reinterpretan los targets bloqueados como fallas del algoritmo.

Son limitaciones de evidencia etiquetada:

- H1 no tiene ambas clases observadas;
- H4/H5 tienen ambas clases, pero no suficiente soporte de grupos para holdout binario;
- H6 sí tiene baseline agrupado y experimento SSL.


In [ ]:
target_status_records = []

for target in ALL_TARGETS:

    readiness_row = target_readiness[
        target_readiness["target"]
        ==
        target
    ]

    continuation_row = baseline_continuation[
        baseline_continuation["target"]
        ==
        target
    ]

    if readiness_row.empty:

        n_0 = np.nan
        n_1 = np.nan
        holdout_support = False

    else:

        row = readiness_row.iloc[0]

        n_0 = row.get(
            "n_0",
            np.nan
        )

        n_1 = row.get(
            "n_1",
            np.nan
        )

        holdout_support = str(
            row.get(
                "grouped_holdout_class_support",
                False
            )
        ).strip().lower() == "true"

    ssl_allowed = False

    if not continuation_row.empty:

        ssl_allowed = str(
            continuation_row.iloc[0].get(
                "semi_supervised_experiment_allowed",
                False
            )
        ).strip().lower() == "true"

    if target == "H6_violation":

        scientific_status = (
            "EXPERIMENTALLY_EVALUATED"
        )

    elif target == "H1_violation":

        scientific_status = (
            "BLOCKED_SINGLE_CLASS"
        )

    else:

        scientific_status = (
            "BLOCKED_INSUFFICIENT_GROUP_SUPPORT"
        )

    target_status_records.append({
        "target":
            target,

        "n_0":
            n_0,

        "n_1":
            n_1,

        "grouped_holdout_support":
            holdout_support,

        "ssl_experiment_allowed":
            ssl_allowed,

        "scientific_status":
            scientific_status
    })

target_status = pd.DataFrame(
    target_status_records
)

display(target_status)


## 5. Comparación principal H6

La decisión principal usa únicamente:

```text
RANDOM_FOREST supervisado
vs
PRIMARY_OVERLAP_095_CAP1
```

Las configuraciones de sensibilidad se reportan después y no se usan para reemplazar retrospectivamente la configuración primaria.


In [ ]:
baseline_h6 = baseline_metrics[
    (
        baseline_metrics["target"]
        ==
        "H6_violation"
    )
    &
    (
        baseline_metrics["model"]
        ==
        "RANDOM_FOREST"
    )
].copy()

primary_h6 = ssl_metrics[
    (
        ssl_metrics["target"]
        ==
        "H6_violation"
    )
    &
    (
        ssl_metrics["config_id"]
        ==
        PRIMARY_CONFIG
    )
].copy()

assert len(baseline_h6) == 1
assert len(primary_h6) == 1

baseline_h6 = baseline_h6.iloc[0]
primary_h6 = primary_h6.iloc[0]

comparison_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision_1",
    "recall_1",
    "f1_1",
    "mcc",
    "roc_auc",
    "pr_auc",
    "brier"
]

primary_comparison_records = []

for metric in comparison_metrics:

    baseline_value = float(
        baseline_h6[metric]
    )

    ssl_value = float(
        primary_h6[metric]
    )

    if metric == "brier":

        favorable_delta = (
            baseline_value
            -
            ssl_value
        )

        direction = (
            "LOWER_IS_BETTER"
        )

    else:

        favorable_delta = (
            ssl_value
            -
            baseline_value
        )

        direction = (
            "HIGHER_IS_BETTER"
        )

    if np.isclose(
        favorable_delta,
        0.0,
        atol=1e-12
    ):

        outcome = "EQUAL"

    elif favorable_delta > 0:

        outcome = "SSL_BETTER"

    else:

        outcome = "BASELINE_BETTER"

    primary_comparison_records.append({
        "metric":
            metric,

        "direction":
            direction,

        "baseline":
            baseline_value,

        "ssl_primary":
            ssl_value,

        "raw_delta_ssl_minus_baseline":
            ssl_value
            -
            baseline_value,

        "favorable_delta":
            favorable_delta,

        "outcome":
            outcome
    })

primary_comparison = pd.DataFrame(
    primary_comparison_records
)

display(primary_comparison)


## 6. Matriz de confusión principal

El baseline y SSL primario usan las mismas 12 filas OOF.


In [ ]:
baseline_h6_oof = baseline_oof[
    (
        baseline_oof["target"]
        ==
        "H6_violation"
    )
    &
    (
        baseline_oof["model"]
        ==
        "RANDOM_FOREST"
    )
][
    [
        "master_instance_id",
        "evaluation_fold",
        "y_true",
        "y_prob",
        "y_pred"
    ]
].rename(
    columns={
        "y_prob":
            "baseline_prob",

        "y_pred":
            "baseline_pred"
    }
)

primary_h6_oof = ssl_oof[
    (
        ssl_oof["target"]
        ==
        "H6_violation"
    )
    &
    (
        ssl_oof["config_id"]
        ==
        PRIMARY_CONFIG
    )
][
    [
        "master_instance_id",
        "evaluation_fold",
        "y_true",
        "y_prob",
        "y_pred"
    ]
].rename(
    columns={
        "y_prob":
            "ssl_prob",

        "y_pred":
            "ssl_pred"
    }
)

paired = baseline_h6_oof.merge(
    primary_h6_oof,
    on=[
        "master_instance_id",
        "evaluation_fold",
        "y_true"
    ],
    how="inner",
    validate="one_to_one"
)

assert len(paired) == 12

paired[
    "baseline_correct"
] = (
    paired[
        "baseline_pred"
    ]
    ==
    paired[
        "y_true"
    ]
)

paired[
    "ssl_correct"
] = (
    paired[
        "ssl_pred"
    ]
    ==
    paired[
        "y_true"
    ]
)

display(paired)

print(
    "Baseline confusion:",
    confusion_matrix(
        paired["y_true"],
        paired["baseline_pred"],
        labels=[0, 1]
    )
)

print(
    "SSL primary confusion:",
    confusion_matrix(
        paired["y_true"],
        paired["ssl_pred"],
        labels=[0, 1]
    )
)


## 7. Comparación pareada de errores

Se cuenta:

```text
b = baseline correcto, SSL incorrecto
c = baseline incorrecto, SSL correcto
```

Se calcula McNemar exacto mediante prueba binomial cuando SciPy está disponible.

Con n=12 esta prueba tiene potencia muy limitada; se reporta como evidencia auxiliar, no como demostración definitiva.


In [ ]:
b = int(
    (
        paired[
            "baseline_correct"
        ]
        &
        ~paired[
            "ssl_correct"
        ]
    ).sum()
)

c = int(
    (
        ~paired[
            "baseline_correct"
        ]
        &
        paired[
            "ssl_correct"
        ]
    ).sum()
)

discordant = b + c

if (
    binomtest is not None
    and
    discordant > 0
):

    mcnemar_p = float(
        binomtest(
            k=min(
                b,
                c
            ),
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue
    )

else:

    mcnemar_p = np.nan

paired_error_test = pd.DataFrame([
    {
        "baseline_correct_ssl_wrong":
            b,

        "baseline_wrong_ssl_correct":
            c,

        "discordant_pairs":
            discordant,

        "mcnemar_exact_p":
            mcnemar_p,

        "interpretation":
            (
                "LOW_POWER_SMALL_N"
            )
    }
])

display(paired_error_test)


## 8. Bootstrap estratificado de diferencias

Para representar la incertidumbre del conjunto extremadamente pequeño, se usa bootstrap estratificado por clase.

Cada réplica remuestrea con reemplazo:

- los 9 negativos;
- los 3 positivos.

Esto preserva la prevalencia observada.

Se estiman intervalos percentiles 95% para:

- balanced accuracy;
- F1;
- MCC;
- ROC-AUC;
- PR-AUC;
- Brier.

Los intervalos son descriptivos del conjunto observado, no intervalos de generalización poblacional robustos.


In [ ]:
rng = np.random.default_rng(
    RANDOM_STATE
)

negative_idx = np.where(
    paired[
        "y_true"
    ].to_numpy()
    ==
    0
)[0]

positive_idx = np.where(
    paired[
        "y_true"
    ].to_numpy()
    ==
    1
)[0]

N_BOOT = 5000

bootstrap_records = []

for iteration in range(
    N_BOOT
):

    sample_idx = np.concatenate(
        [
            rng.choice(
                negative_idx,
                size=len(
                    negative_idx
                ),
                replace=True
            ),

            rng.choice(
                positive_idx,
                size=len(
                    positive_idx
                ),
                replace=True
            )
        ]
    )

    sample = paired.iloc[
        sample_idx
    ]

    y_true = sample[
        "y_true"
    ].to_numpy()

    baseline_values = compute_metrics(
        y_true,
        sample[
            "baseline_pred"
        ].to_numpy(),
        sample[
            "baseline_prob"
        ].to_numpy()
    )

    ssl_values = compute_metrics(
        y_true,
        sample[
            "ssl_pred"
        ].to_numpy(),
        sample[
            "ssl_prob"
        ].to_numpy()
    )

    bootstrap_records.append({
        "balanced_accuracy_delta":
            ssl_values[
                "balanced_accuracy"
            ]
            -
            baseline_values[
                "balanced_accuracy"
            ],

        "f1_delta":
            ssl_values[
                "f1_1"
            ]
            -
            baseline_values[
                "f1_1"
            ],

        "mcc_delta":
            ssl_values[
                "mcc"
            ]
            -
            baseline_values[
                "mcc"
            ],

        "roc_auc_delta":
            ssl_values[
                "roc_auc"
            ]
            -
            baseline_values[
                "roc_auc"
            ],

        "pr_auc_delta":
            ssl_values[
                "pr_auc"
            ]
            -
            baseline_values[
                "pr_auc"
            ],

        # Brier: positivo = mejora SSL
        "brier_favorable_delta":
            baseline_values[
                "brier"
            ]
            -
            ssl_values[
                "brier"
            ]
    })

bootstrap = pd.DataFrame(
    bootstrap_records
)

bootstrap_ci_records = []

for column in bootstrap.columns:

    values = bootstrap[
        column
    ].dropna()

    bootstrap_ci_records.append({
        "metric_delta":
            column,

        "mean":
            float(
                values.mean()
            ),

        "median":
            float(
                values.median()
            ),

        "ci95_low":
            float(
                values.quantile(
                    0.025
                )
            ),

        "ci95_high":
            float(
                values.quantile(
                    0.975
                )
            ),

        "proportion_gt_0":
            float(
                (
                    values
                    >
                    0
                ).mean()
            )
    })

bootstrap_ci = pd.DataFrame(
    bootstrap_ci_records
)

display(bootstrap_ci)


## 9. Diagnóstico del pseudo-etiquetado primario

Se revisa:

- tamaño de U;
- U elegible por fold;
- número de pseudo-etiquetas;
- balance pseudo 0/1;
- estabilidad cuando una instancia fue seleccionada más de una vez.

La ausencia de pseudo-etiquetas positivas es una señal relevante porque puede desplazar el modelo hacia la clase 0.


In [ ]:
primary_pseudo_summary = pseudo_summary[
    pseudo_summary[
        "config_id"
    ]
    ==
    PRIMARY_CONFIG
].copy()

primary_pseudo_labels = pseudo_labels[
    pseudo_labels[
        "config_id"
    ]
    ==
    PRIMARY_CONFIG
].copy()

primary_stability = pseudo_stability[
    pseudo_stability[
        "config_id"
    ]
    ==
    PRIMARY_CONFIG
].copy()

unique_primary_pseudo = (
    primary_pseudo_labels[
        "master_instance_id"
    ].nunique()
    if not primary_pseudo_labels.empty
    else 0
)

multi_fold_selected = (
    int(
        (
            primary_stability[
                "folds_selected"
            ]
            >
            1
        ).sum()
    )
    if not primary_stability.empty
    else 0
)

fully_stable_multi_fold = (
    int(
        (
            (
                primary_stability[
                    "folds_selected"
                ]
                >
                1
            )
            &
            (
                primary_stability[
                    "fully_stable_when_selected"
                ].astype(str)
                .str.lower()
                .eq("true")
            )
        ).sum()
    )
    if not primary_stability.empty
    else 0
)

pseudo_diagnostics = pd.DataFrame([
    {
        "config_id":
            PRIMARY_CONFIG,

        "U_total":
            int(
                primary_pseudo_summary[
                    "U_total"
                ].iloc[0]
            ),

        "mean_eligible_U_per_fold":
            float(
                primary_pseudo_summary[
                    "mean_eligible_U_per_fold"
                ].iloc[0]
            ),

        "pseudo_selection_events":
            int(
                primary_pseudo_summary[
                    "total_pseudo_selection_events"
                ].iloc[0]
            ),

        "unique_pseudo_instances":
            unique_primary_pseudo,

        "pseudo_0_events":
            int(
                primary_pseudo_summary[
                    "pseudo_0_events"
                ].iloc[0]
            ),

        "pseudo_1_events":
            int(
                primary_pseudo_summary[
                    "pseudo_1_events"
                ].iloc[0]
            ),

        "instances_selected_multiple_folds":
            multi_fold_selected,

        "fully_stable_multi_fold_instances":
            fully_stable_multi_fold
    }
])

display(pseudo_diagnostics)


## 10. Análisis de sensibilidad

Se comparan todas las configuraciones, pero no se elige retrospectivamente la de mejor test.

Una configuración que selecciona cero pseudo-etiquetas y reproduce el baseline no constituye evidencia de beneficio semisupervisado.


In [ ]:
sensitivity = (
    ssl_vs_baseline.copy()
)

sensitivity = sensitivity.merge(
    pseudo_summary[
        [
            "config_id",
            "total_pseudo_selection_events",
            "pseudo_0_events",
            "pseudo_1_events",
            "mean_eligible_U_per_fold",
            "mean_selected_pseudo_per_fold"
        ]
    ],
    on="config_id",
    how="left",
    validate="one_to_one"
)

def classify_sensitivity_row(row):

    pseudo_events = int(
        row[
            "total_pseudo_selection_events"
        ]
    )

    delta_bal = float(
        row[
            "delta_balanced_accuracy"
        ]
    )

    delta_f1 = float(
        row[
            "delta_f1"
        ]
    )

    delta_mcc = float(
        row[
            "delta_mcc"
        ]
    )

    if pseudo_events == 0:

        return (
            "NO_SSL_UPDATE_BASELINE_REPRODUCED"
        )

    if (
        delta_bal > 0
        and
        delta_f1 >= 0
        and
        delta_mcc >= 0
    ):

        return (
            "THRESHOLD_METRICS_IMPROVED"
        )

    if (
        delta_bal < 0
        or
        delta_f1 < 0
        or
        delta_mcc < 0
    ):

        return (
            "THRESHOLD_METRICS_DETERIORATED"
        )

    return (
        "NO_CLEAR_THRESHOLD_CHANGE"
    )


sensitivity[
    "sensitivity_interpretation"
] = sensitivity.apply(
    classify_sensitivity_row,
    axis=1
)

display(
    sensitivity[
        [
            "config_id",
            "selection_role",
            "total_pseudo_selection_events",
            "pseudo_0_events",
            "pseudo_1_events",
            "delta_balanced_accuracy",
            "delta_f1",
            "delta_mcc",
            "delta_roc_auc",
            "delta_pr_auc",
            "delta_brier",
            "sensitivity_interpretation"
        ]
    ]
)


## 11. Auditoría de procedencia de etiquetas

La procedencia no fue predictor, pero está relacionada con la clase observada.

Esto limita la interpretación porque:

- A/B/C no representan la misma fuerza de correspondencia fuente;
- el conjunto es muy pequeño;
- una asociación entre procedencia y clase puede confundir el patrón aprendido.


In [ ]:
display(
    baseline_provenance
)

provenance_limitation = pd.DataFrame([
    {
        "issue":
            "SOURCE_MATCH_LEVEL_ASSOCIATED_WITH_CLASS",

        "present":
            True,

        "implication":
            (
                "The labeled H6 seed mixes A/B/C "
                "source-match evidence and the distribution "
                "is not independent of class."
            )
    }
])

display(provenance_limitation)


## 12. Decisión científica H6

Criterio conservador:

### `SSL_IMPROVES`

Requiere que la configuración primaria mejore:

- balanced accuracy;
- F1 o MCC;

sin deterioro material de las métricas principales.

### `SSL_NOT_BENEFICIAL`

Si la configuración primaria deteriora balanced accuracy/F1/MCC y las configuraciones que realmente agregan pseudo-etiquetas no muestran mejora robusta.

### `INCONCLUSIVE`

Si existen señales contradictorias sin una dirección consistente.

La configuración de sensibilidad 0.99 que selecciona cero pseudo-etiquetas se interpreta como reproducción del baseline, no como mejora SSL.


In [ ]:
primary_row = ssl_vs_baseline[
    ssl_vs_baseline[
        "config_id"
    ]
    ==
    PRIMARY_CONFIG
].iloc[0]

primary_pseudo_events = int(
    pseudo_diagnostics[
        "pseudo_selection_events"
    ].iloc[0]
)

primary_threshold_deterioration = bool(
    primary_row[
        "delta_balanced_accuracy"
    ]
    <
    0
    or
    primary_row[
        "delta_f1"
    ]
    <
    0
    or
    primary_row[
        "delta_mcc"
    ]
    <
    0
)

real_ssl_rows = sensitivity[
    sensitivity[
        "total_pseudo_selection_events"
    ]
    >
    0
].copy()

any_real_ssl_threshold_improvement = bool(
    (
        (
            real_ssl_rows[
                "delta_balanced_accuracy"
            ]
            >
            0
        )
        &
        (
            real_ssl_rows[
                "delta_f1"
            ]
            >=
            0
        )
        &
        (
            real_ssl_rows[
                "delta_mcc"
            ]
            >=
            0
        )
    ).any()
)

if (
    primary_pseudo_events
    >
    0
    and
    primary_threshold_deterioration
    and
    not any_real_ssl_threshold_improvement
):

    h6_ssl_conclusion = (
        "SSL_NOT_BENEFICIAL_WITH_CURRENT_EVIDENCE"
    )

elif any_real_ssl_threshold_improvement:

    h6_ssl_conclusion = (
        "SSL_IMPROVEMENT_SIGNAL_REQUIRES_CONFIRMATION"
    )

else:

    h6_ssl_conclusion = (
        "SSL_RESULT_INCONCLUSIVE"
    )

h6_decision = pd.DataFrame([
    {
        "target":
            "H6_violation",

        "supervised_baseline_available":
            True,

        "ssl_primary_pseudo_events":
            primary_pseudo_events,

        "primary_delta_balanced_accuracy":
            float(
                primary_row[
                    "delta_balanced_accuracy"
                ]
            ),

        "primary_delta_f1":
            float(
                primary_row[
                    "delta_f1"
                ]
            ),

        "primary_delta_mcc":
            float(
                primary_row[
                    "delta_mcc"
                ]
            ),

        "primary_delta_roc_auc":
            float(
                primary_row[
                    "delta_roc_auc"
                ]
            ),

        "primary_delta_pr_auc":
            float(
                primary_row[
                    "delta_pr_auc"
                ]
            ),

        "primary_delta_brier":
            float(
                primary_row[
                    "delta_brier"
                ]
            ),

        "ssl_conclusion":
            h6_ssl_conclusion
    }
])

display(h6_decision)


## 13. Decisión global del proyecto de ML

La meta original era producir cuatro clasificadores:

```text
H1
H4
H5
H6
```

Con la evidencia actual:

- H1 no puede evaluarse binariamente;
- H4/H5 no tienen soporte suficiente para holdout agrupado;
- H6 sí fue evaluado;
- el SSL primario de H6 no supera al baseline supervisado en métricas de clasificación al threshold 0.5.

Por tanto, no existe todavía evidencia para afirmar que se dispone de un modelo semisupervisado validado para las cuatro heurísticas.


In [ ]:
project_decision = pd.DataFrame([
    {
        "component":
            "H1_violation",

        "status":
            "NOT_EVALUABLE_CURRENT_L",

        "deployment_candidate":
            False
    },

    {
        "component":
            "H4_violation",

        "status":
            "BINARY_SEED_ONLY_HOLDOUT_BLOCKED",

        "deployment_candidate":
            False
    },

    {
        "component":
            "H5_violation",

        "status":
            "BINARY_SEED_ONLY_HOLDOUT_BLOCKED",

        "deployment_candidate":
            False
    },

    {
        "component":
            "H6_violation",

        "status":
            (
                "SUPERVISED_BASELINE_EXPERIMENTAL_ONLY"
                if h6_ssl_conclusion
                ==
                "SSL_NOT_BENEFICIAL_WITH_CURRENT_EVIDENCE"
                else
                "EXPERIMENTAL_RESULT_REQUIRES_CONFIRMATION"
            ),

        "deployment_candidate":
            False
    },

    {
        "component":
            "FULL_H1_H4_H5_H6_MODEL",

        "status":
            "NOT_READY_FOR_SCIENTIFIC_DEPLOYMENT",

        "deployment_candidate":
            False
    }
])

display(project_decision)


## 14. Redacción metodológica para la tesis

El texto se genera como apoyo y debe integrarse en el capítulo de resultados/discusión junto con las tablas numéricas.


In [ ]:
thesis_conclusion = (
    "El conjunto maestro semisupervisado permitió realizar una evaluación "
    "experimental para la heurística H6, mientras que H1, H4 y H5 "
    "permanecieron restringidas por la disponibilidad y distribución de las "
    "etiquetas observadas. Para H6 se construyó un baseline supervisado mediante "
    "Random Forest y se comparó con un esquema de self-training que incorporó "
    "instancias no etiquetadas mediante pseudo-etiquetas de alta confianza. "
    "La configuración semisupervisada primaria no mejoró el desempeño de "
    "clasificación respecto al baseline supervisado en balanced accuracy, F1 y "
    "MCC. Aunque algunas métricas basadas en probabilidades mostraron variaciones "
    "favorables, las pseudo-etiquetas seleccionadas fueron exclusivamente de la "
    "clase negativa, lo que produjo un desplazamiento del clasificador y una "
    "reducción de la recuperación de violaciones H6. Las configuraciones de "
    "sensibilidad que incorporaron pseudo-etiquetas tampoco mostraron una mejora "
    "consistente en las métricas principales. En consecuencia, con la evidencia "
    "actual no se sostiene que el aprendizaje semisupervisado mejore el baseline "
    "supervisado. Este resultado se considera experimental debido al reducido "
    "número de instancias etiquetadas, al uso de un espacio de características "
    "estructurales proxy y al cambio de dominio entre las fuentes web etiquetadas "
    "y las interfaces Android no etiquetadas."
)

print(thesis_conclusion)


## 15. Guardar artefactos finales

In [ ]:
primary_comparison.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_primary_vs_baseline.csv",
    index=False
)

paired.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_paired_predictions.csv",
    index=False
)

paired_error_test.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_paired_error_test.csv",
    index=False
)

bootstrap_ci.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_bootstrap_ci.csv",
    index=False
)

pseudo_diagnostics.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_pseudo_label_diagnostics.csv",
    index=False
)

sensitivity.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_sensitivity.csv",
    index=False
)

target_status.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_target_status.csv",
    index=False
)

h6_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_h6_decision.csv",
    index=False
)

project_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_project_decision.csv",
    index=False
)

with open(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_thesis_conclusion.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        thesis_conclusion
    )

with open(
    SEMISUP_REPORTS_DIR
    / "model_evaluation_final_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "primary_target":
                "H6_violation",

            "primary_ssl_config":
                PRIMARY_CONFIG,

            "h6_ssl_conclusion":
                h6_ssl_conclusion,

            "full_four_target_model_ready":
                False,

            "recommended_current_model_status":
                "SUPERVISED_BASELINE_EXPERIMENTAL_ONLY",

            "limitations": [
                "very_small_labeled_set",
                "proxy_feature_space",
                "web_android_domain_shift",
                "mixed_source_match_provenance",
                "H1_H4_H5_not_fully_evaluable"
            ]
        },
        file,
        indent=2
    )


In [ ]:
expected_outputs = [
    "model_evaluation_primary_vs_baseline.csv",
    "model_evaluation_paired_predictions.csv",
    "model_evaluation_paired_error_test.csv",
    "model_evaluation_bootstrap_ci.csv",
    "model_evaluation_pseudo_label_diagnostics.csv",
    "model_evaluation_sensitivity.csv",
    "model_evaluation_target_status.csv",
    "model_evaluation_h6_decision.csv",
    "model_evaluation_project_decision.csv",
    "model_evaluation_thesis_conclusion.txt",
    "model_evaluation_final_summary.json"
]

for filename in expected_outputs:

    path = (
        SEMISUP_REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


## Cierre de la ruta 24–30

La ruta:

```text
24_semisupervised_master_dataset_design
25_semisupervised_feature_space
26_semisupervised_labeled_unlabeled_split
27_semisupervised_trainability_audit
28_supervised_baseline
29_semisupervised_training
30_model_evaluation
```

queda metodológicamente cerrada cuando este notebook se ejecuta sin errores.

### Resultado que debe conservarse aunque sea negativo

Un experimento semisupervisado **no tiene que mejorar** al baseline para ser científicamente válido.

Si la evidencia muestra deterioro, ese es el resultado que debe reportarse.

La conclusión actual no debe expresarse como:

> “el aprendizaje semisupervisado funciona para las cuatro heurísticas”.

Debe expresarse como:

> “se evaluó la viabilidad de un enfoque semisupervisado con datasets públicos heterogéneos; únicamente H6 alcanzó soporte mínimo para evaluación agrupada y, bajo la configuración primaria estudiada, el pseudo-etiquetado no mejoró el baseline supervisado”.

El paso posterior ya no es otro notebook de entrenamiento automático. A partir de aquí corresponde decidir metodológicamente si:

- se amplía la recuperación de evidencia etiquetada;
- se desarrolla la extensión con el modelo H6 únicamente como prototipo experimental;
- o se redefine el esquema de validación antes de la implementación final.
